In [3]:
%pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
corpus = [
    "AI detects algae in water",
    "AI predicts pollution in rivers",
    "IoT sensors for algae detection",
    "AI for image recognition",
    "Satellite data for lakes"
]

query = "AI algae detection"

In [ ]:
from rank_bm25 import BM25Okapi

# whitespace tokenizer
tokenized_corpus = [doc.lower().split() for doc in corpus]
tokenized_query = query.lower().split()

In [15]:
#Initialize BM25 and get scores
bm25 = BM25Okapi(tokenized_corpus)
scores = bm25.get_scores(tokenized_query)

In [16]:
for i, s in enumerate(scores):
    print(f"Doc {i+1} → Score: {s:.3f} | {corpus[i]}")

Doc 1 → Score: 0.526 | AI detects algae in water
Doc 2 → Score: 0.202 | AI predicts pollution in rivers
Doc 3 → Score: 1.381 | IoT sensors for algae detection
Doc 4 → Score: 0.223 | AI for image recognition
Doc 5 → Score: 0.000 | Satellite data for lakes


In [17]:
import numpy as np

ranking = np.argsort(scores)[::-1]
print("\nRanked Documents:")
for rank, idx in enumerate(ranking, start=1):
    print(f"{rank}. ({scores[idx]:.3f}) {corpus[idx]}")


Ranked Documents:
1. (1.381) IoT sensors for algae detection
2. (0.526) AI detects algae in water
3. (0.223) AI for image recognition
4. (0.202) AI predicts pollution in rivers
5. (0.000) Satellite data for lakes


In [18]:
%pip install PyPDF2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### From actual PDFs

In [2]:
import PyPDF2
from rank_bm25 import BM25Okapi

# STEP 1: Extract Text by Sections (Pages)

def extract_pdf_by_pages(pdf_path):
    """Opens a PDF and extracts text, keeping each page as a separate section."""
    pages_data = []
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page_num, page in enumerate(reader.pages):
                text = page.extract_text()
                if text:
                    pages_data.append((page_num + 1, text)) 
    except FileNotFoundError:
        print(f"Error: Could not find {pdf_path}")
    return pages_data

pdf_files = ["Natural-Language-Processing-Python.pdf", "learning-langchain-for-true-epub-9781098167288.pdf", "test.pdf"]

print("Extracting and structuring PDF data\n")
pdf_structured_data = {pdf: extract_pdf_by_pages(pdf) for pdf in pdf_files}


# Independent Queries

queries = [
    "what is a tokenization",
    "what are the different types of tokenizers"
]


doc_corpus = []
for pdf in pdf_files:
    full_text = " ".join([text for page_num, text in pdf_structured_data[pdf]])
    doc_corpus.append(full_text.lower().split())

bm25_docs = BM25Okapi(doc_corpus)


# Loop and Score Each Query

for raw_query in queries:

    print(f" EVALUATING QUERY: '{raw_query}'")

    tokenized_query = raw_query.lower().split()
    
    # Pass 1: Find the Best Document
    doc_scores = bm25_docs.get_scores(tokenized_query)
    best_doc_score = max(doc_scores)
    
    # Safety check: if none of the words exist in any document
    if best_doc_score == 0.0:
        print("Result: No matching documents found for this query.\n")
        continue

    # Identify the winner
    best_doc_index = list(doc_scores).index(best_doc_score)
    winning_pdf = pdf_files[best_doc_index]

    print(f" Best Overall Document: {winning_pdf} (Score: {best_doc_score:.4f})")

    # Pass 2: Find the Best Section
    winning_pages = pdf_structured_data[winning_pdf]
    page_corpus = [text.lower().split() for page_num, text in winning_pages]

    bm25_pages = BM25Okapi(page_corpus)
    page_scores = bm25_pages.get_scores(tokenized_query)

    best_page_score = max(page_scores)
    best_page_index = list(page_scores).index(best_page_score)

    winning_page_num, winning_page_text = winning_pages[best_page_index]

    print(f" Best Section Found: Page {winning_page_num} (Score: {best_page_score:.4f})")
    print("\nSection Snippet ")
    print(winning_page_text[:1000] + "...\n")

Extracting and structuring PDF data

 EVALUATING QUERY: 'what is a tokenization'
 Best Overall Document: test.pdf (Score: 0.6330)
 Best Section Found: Page 84 (Score: 9.5329)

Section Snippet 
NOTE
What is the significance of  whitespace characters? These are important for models to
understand or generate code. A model that uses a single token to represent four
consecutive whitespace characters is more tuned to a Python code dataset. While a
model can live with representing it as four dif ferent tokens, it does make the modeling
more dif ficult as the model needs to keep track of the indentation level, which often
leads to worse performance. This is an example of where tokenization choices can help
the model improve on a certain task.
Flan-T5 (2022)
Tokenization  method: Flan-T5  uses a tokenizer implementation called
SentencePiece, introduced in “SentencePiece: A simple and language
independent subword tokenizer and detokenizer for neural text processing” ,
which supports BPE and the 